# High-z Example 24: Schema Self-Description Test

**EPS Research — High-z Kinematic Corpus Z1**

The Z1 corpus is designed to be self-describing for LLM consumption.
This example verifies that all field names, units, and quality flags
are explicit and unambiguous in the JSON schema.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
# Verify schema self-description
print("Z1 Corpus Schema Verification")
print("="*50)
print(f"\nTop-level keys: {list(corpus.keys())}")
if corpus.get('metadata'):
    print(f"\nMetadata:")
    for k, v in corpus['metadata'].items():
        print(f"  {k}: {v}")

print(f"\nPer-galaxy fields:")
g = galaxies[0]
for k, v in g.items():
    if k != 'data':
        print(f"  {k}: {type(v).__name__} = {v}")

print(f"\nPer-ring fields (tier-1):")
tier1 = next(g for g in galaxies if g.get('quality_tier')==1)
if tier1.get('data'):
    for k, v in tier1['data'][0].items():
        print(f"  {k}: {type(v).__name__} = {v}")

print(f"\nQuality tier system:")
print(f"  Tier 1: per-ring Vrot, sigma, Mdyn ({sum(1 for g in galaxies if g.get('quality_tier')==1)} galaxies)")
print(f"  Tier 2: morpho-kinematic classification only ({sum(1 for g in galaxies if g.get('quality_tier')==2)} galaxies)")